In [1]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("../data/raw/UserBehavior.csv")
DEV_PATH = Path("../data/interim/UserBehavior_dev.csv")

columns = [
    "user_id",
    "item_id",
    "category_id",
    "behavior_type",
    "timestamp"
]

CHUNK_SIZE = 1_000_000

In [2]:
reader = pd.read_csv(
    RAW_PATH,
    names=columns,
    chunksize=CHUNK_SIZE
)

first_chunk = next(reader)

print(first_chunk.shape)
first_chunk.head()

(1000000, 5)


,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


In [3]:
# 有效业务时间范围
START_TS = int(
    pd.Timestamp(
        "2017-11-25 00:00:00",
        tz="Asia/Shanghai"
    ).timestamp()
)

END_TS = int(
    pd.Timestamp(
        "2017-12-04 00:00:00",
        tz="Asia/Shanghai"
    ).timestamp()
)

# 防止重复运行时，把数据重复追加进旧文件
if DEV_PATH.exists():
    DEV_PATH.unlink()

reader = pd.read_csv(
    RAW_PATH,
    names=columns,
    chunksize=CHUNK_SIZE
)

total_rows = 0
dev_rows = 0

for chunk_id, chunk in enumerate(reader, start=1):

    total_rows += len(chunk)

    # 1. 只保留有效时间
    time_mask = (
        (chunk["timestamp"] >= START_TS)
        &
        (chunk["timestamp"] < END_TS)
    )

    # 2. 按用户抽约 1%
    user_mask = chunk["user_id"] % 100 == 0

    dev_chunk = chunk.loc[
        time_mask & user_mask
    ]

    dev_rows += len(dev_chunk)

    # 分块写入，不全部堆进内存
    dev_chunk.to_csv(
        DEV_PATH,
        mode="a",
        index=False,
        header=(chunk_id == 1)
    )

    print(
        f"Chunk {chunk_id}: "
        f"processed={total_rows:,}, "
        f"dev_rows={dev_rows:,}"
    )

Chunk 1: processed=1,000,000, dev_rows=10,004
Chunk 2: processed=2,000,000, dev_rows=19,118
Chunk 3: processed=3,000,000, dev_rows=28,594
Chunk 4: processed=4,000,000, dev_rows=38,195
Chunk 5: processed=5,000,000, dev_rows=49,097
Chunk 6: processed=6,000,000, dev_rows=60,327
Chunk 7: processed=7,000,000, dev_rows=69,874
Chunk 8: processed=8,000,000, dev_rows=79,807
Chunk 9: processed=9,000,000, dev_rows=87,185
Chunk 10: processed=10,000,000, dev_rows=96,354
Chunk 11: processed=11,000,000, dev_rows=107,091
Chunk 12: processed=12,000,000, dev_rows=116,977
Chunk 13: processed=13,000,000, dev_rows=127,396
Chunk 14: processed=14,000,000, dev_rows=137,641
Chunk 15: processed=15,000,000, dev_rows=146,719
Chunk 16: processed=16,000,000, dev_rows=157,107
Chunk 17: processed=17,000,000, dev_rows=168,608
Chunk 18: processed=18,000,000, dev_rows=179,158
Chunk 19: processed=19,000,000, dev_rows=187,150
Chunk 20: processed=20,000,000, dev_rows=197,904
Chunk 21: processed=21,000,000, dev_rows=206,210

In [4]:
dev_df = pd.read_csv(DEV_PATH)

print("DEV shape:")
print(dev_df.shape)

print("\nUnique users:")
print(dev_df["user_id"].nunique())

print("\nUnique items:")
print(dev_df["item_id"].nunique())

print("\nUnique categories:")
print(dev_df["category_id"].nunique())

print("\nBehavior counts:")
print(dev_df["behavior_type"].value_counts())

print("\nMissing values:")
print(dev_df.isna().sum())

print("\nDuplicate rows:")
print(dev_df.duplicated().sum())

DEV shape:
(1001832, 5)

Unique users:
9895

Unique items:
399480

Unique categories:
5761

Behavior counts:
behavior_type
pv      897440
cart     56377
fav      28029
buy      19986
Name: count, dtype: int64

Missing values:
user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
dtype: int64

Duplicate rows:
0


In [5]:
print("All users satisfy sampling rule:")
print((dev_df["user_id"] % 100 == 0).all())

print("\nExample user ids:")
print(dev_df["user_id"].unique()[:20])

All users satisfy sampling rule:
True

Example user ids:
[    100    1000 1001000 1001500 1001700 1001800 1002200 1003200  100400
 1004800  100600 1006400 1006900  100700 1007100 1007200  100800 1008100
 1008500 1009000]


In [6]:
dev_df["datetime"] = (
    pd.to_datetime(
        dev_df["timestamp"],
        unit="s",
        utc=True
    )
    .dt.tz_convert("Asia/Shanghai")
)

print("DEV earliest time:")
print(dev_df["datetime"].min())

print("\nDEV latest time:")
print(dev_df["datetime"].max())

DEV earliest time:
2017-11-25 00:00:00+08:00

DEV latest time:
2017-12-03 23:59:59+08:00


In [7]:
# 提取日期
dev_df["date"] = dev_df["datetime"].dt.date

# 每天各种行为数量
daily_behavior = pd.crosstab(
    dev_df["date"],
    dev_df["behavior_type"]
)

daily_behavior["total"] = daily_behavior.sum(axis=1)

print("Daily behavior distribution:")
print(daily_behavior)
daily_users = (
    dev_df.groupby("date")["user_id"]
          .nunique()
)

print("\nDaily active users:")
print(daily_users)
daily_buy_stats = (
    dev_df[dev_df["behavior_type"] == "buy"]
    .groupby("date")
    .agg(
        buy_count=("item_id", "size"),
        buy_users=("user_id", "nunique"),
        buy_items=("item_id", "nunique")
    )
)

print("\nDaily buy statistics:")
print(daily_buy_stats)

Daily behavior distribution:
behavior_type   buy  cart   fav      pv   total
date                                           
2017-11-25     1941  5678  2925   92090  102634
2017-11-26     2109  5814  3078   96101  107102
2017-11-27     2303  5659  2861   90134  100957
2017-11-28     2077  5517  2669   88002   98265
2017-11-29     2195  5454  3007   92903  103559
2017-11-30     2128  5788  2849   92751  103516
2017-12-01     2054  6331  2978   97489  108852
2017-12-02     2599  8095  3887  123825  138406
2017-12-03     2580  8041  3775  124145  138541

Daily active users:
date
2017-11-25    7062
2017-11-26    7198
2017-11-27    7151
2017-11-28    7030
2017-11-29    7174
2017-11-30    7305
2017-12-01    7420
2017-12-02    9719
2017-12-03    9688
Name: user_id, dtype: int64

Daily buy statistics:
            buy_count  buy_users  buy_items
date                                       
2017-11-25       1941       1271       1876
2017-11-26       2109       1370       2039
2017-11-27       23

In [8]:
train_df = dev_df[
    dev_df["date"] <= pd.to_datetime("2017-12-01").date()
].copy()

val_df = dev_df[
    dev_df["date"] == pd.to_datetime("2017-12-02").date()
].copy()

test_df = dev_df[
    dev_df["date"] == pd.to_datetime("2017-12-03").date()
].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (724885, 7)
Validation: (138406, 7)
Test: (138541, 7)


In [9]:
test_buy = test_df[
    test_df["behavior_type"] == "buy"
].copy()

print("Test buy interactions:")
print(len(test_buy))

print("\nTest buyers:")
print(test_buy["user_id"].nunique())

print("\nTest purchased items:")
print(test_buy["item_id"].nunique())

Test buy interactions:
2580

Test buyers:
1756

Test purchased items:
2450


In [10]:
train_users = set(train_df["user_id"].unique())
train_items = set(train_df["item_id"].unique())

test_buy["warm_user"] = test_buy["user_id"].isin(train_users)
test_buy["warm_item"] = test_buy["item_id"].isin(train_items)

print("Warm user ratio:")
print(test_buy["warm_user"].mean())

print("\nWarm item ratio:")
print(test_buy["warm_item"].mean())

print("\nFully warm interaction ratio:")
print(
    (
        test_buy["warm_user"]
        &
        test_buy["warm_item"]
    ).mean()
)

Warm user ratio:
1.0

Warm item ratio:
0.6170542635658914

Fully warm interaction ratio:
0.6170542635658914


In [11]:
val_buy = val_df[
    val_df["behavior_type"] == "buy"
].copy()

val_buy["warm_user"] = val_buy["user_id"].isin(train_users)
val_buy["warm_item"] = val_buy["item_id"].isin(train_items)

print("Validation buy interactions:")
print(len(val_buy))

print("\nValidation buyers:")
print(val_buy["user_id"].nunique())

print("\nValidation warm user ratio:")
print(val_buy["warm_user"].mean())

print("\nValidation warm item ratio:")
print(val_buy["warm_item"].mean())

print("\nValidation fully warm ratio:")
print(
    (
        val_buy["warm_user"]
        &
        val_buy["warm_item"]
    ).mean()
)

Validation buy interactions:
2599

Validation buyers:
1760

Validation warm user ratio:
1.0

Validation warm item ratio:
0.6479415159676799

Validation fully warm ratio:
0.6479415159676799


In [12]:
test_buy_user_item_duplicates = (
    test_buy.duplicated(
        subset=["user_id", "item_id"]
    ).sum()
)

val_buy_user_item_duplicates = (
    val_buy.duplicated(
        subset=["user_id", "item_id"]
    ).sum()
)

print("Validation repeated user-item purchases:")
print(val_buy_user_item_duplicates)

print("\nTest repeated user-item purchases:")
print(test_buy_user_item_duplicates)

Validation repeated user-item purchases:
59

Test repeated user-item purchases:
68


In [13]:
train_val_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

train_val_users = set(
    train_val_df["user_id"].unique()
)

train_val_items = set(
    train_val_df["item_id"].unique()
)

test_buy["warm_user_final"] = (
    test_buy["user_id"].isin(train_val_users)
)

test_buy["warm_item_final"] = (
    test_buy["item_id"].isin(train_val_items)
)

print("Final-test warm user ratio:")
print(test_buy["warm_user_final"].mean())

print("\nFinal-test warm item ratio:")
print(test_buy["warm_item_final"].mean())

print("\nFinal-test fully warm ratio:")
print(
    (
        test_buy["warm_user_final"]
        &
        test_buy["warm_item_final"]
    ).mean()
)

Final-test warm user ratio:
1.0

Final-test warm item ratio:
0.6906976744186046

Final-test fully warm ratio:
0.6906976744186046


In [14]:
val_ground_truth = (
    val_buy[
        ["user_id", "item_id"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

test_ground_truth = (
    test_buy[
        ["user_id", "item_id"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Validation unique user-item targets:")
print(len(val_ground_truth))

print("\nTest unique user-item targets:")
print(len(test_ground_truth))

Validation unique user-item targets:
2540

Test unique user-item targets:
2512


In [15]:
PROCESSED_DIR = Path("../data/processed/dev")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [16]:
MODEL_COLUMNS = [
    "user_id",
    "item_id",
    "category_id",
    "behavior_type",
    "timestamp"
]

train_df[MODEL_COLUMNS].to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

val_df[MODEL_COLUMNS].to_csv(
    PROCESSED_DIR / "validation.csv",
    index=False
)

test_df[MODEL_COLUMNS].to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)

val_ground_truth.to_csv(
    PROCESSED_DIR / "validation_ground_truth.csv",
    index=False
)

test_ground_truth.to_csv(
    PROCESSED_DIR / "test_ground_truth.csv",
    index=False
)

In [17]:
for file_path in sorted(PROCESSED_DIR.glob("*.csv")):
    temp_df = pd.read_csv(file_path)

    print(
        file_path.name,
        "->",
        temp_df.shape
    )

test.csv -> (138541, 5)
test_ground_truth.csv -> (2512, 2)
train.csv -> (724885, 5)
validation.csv -> (138406, 5)
validation_ground_truth.csv -> (2540, 2)
